In [3]:
cd ..

/Users/RGG7/PycharmProjects/fields-of-study-pipeline


In [4]:
from fos.vectors import load_field_fasttext, load_field_keys, load_field_entities, load_field_tfidf

In [5]:
from fos.entity import embed_entities

In [6]:
from fos.model import FieldModel

In [7]:
from fos.settings import CORPUS_DIR, ASSETS_DIR

In [8]:
from fos.util import iter_bq_extract

In [9]:
from fos.vectors import batch_sparse_similarity, sparse_norm

In [10]:
import pandas as pd
import numpy as np

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

In [94]:
import json

In [12]:
keys = load_field_keys('en')

In [13]:
levels = pd.read_json('assets/fields/field_meta.jsonl', lines=True)
levels = levels.rename(columns={'name': 'display_name'})

In [14]:
children = pd.read_json('assets/fields/field_children.jsonl', lines=True)

In [15]:
model = FieldModel()
field_fasttext = model.field_fasttext.index
field_tfidf = model.field_tfidf.index
dictionary = model.dictionary
field_entities = model.field_entities.index

In [16]:
meta = pd.read_json(ASSETS_DIR / 'fields/field_meta.jsonl', lines=True)
index = meta['name'].to_numpy()
levels = meta['level'].to_numpy()

In [17]:
l0_fasttext = field_fasttext[levels == 0]

In [18]:
l1_fasttext = field_fasttext[levels == 1]

In [19]:
l0_tfidf = field_tfidf[levels == 0]
l0_entity = field_entities[levels == 0]
l1_tfidf = field_tfidf[levels == 1]
l1_entity = field_entities[levels == 1]

In [20]:
l0_fasttext_comp = field_fasttext[levels == 0]
l0_tfidf_comp = field_tfidf[levels == 0]
l0_entity_comp = field_entities[levels == 0]

In [21]:
def cosine_similarity(fieldsa, fieldsb):
    """Calculate the similarity between fields."""
    return np.dot(fieldsa, fieldsb.T)

In [22]:
ft_sim = cosine_similarity(l0_fasttext, l0_fasttext_comp)

In [23]:
from gensim import matutils
field_corpus = matutils.Sparse2Corpus(l0_tfidf)

In [24]:
entity_sim = cosine_similarity(l0_entity, l0_entity_comp)

In [25]:
def average_similarity(ft, entity, tfidf):
    """Average the FastText, entity FastText, and tf-idf similarities for a batch.

    Not all documents will have similarity scores of all types. We take the average
    over available score types.
    """
    sims = np.array((ft, tfidf.A, entity))
    valid_sums = np.sum(sims, axis=0)
    valid_counts = 3
    avg = valid_sums / 3
    return avg

In [26]:
def sparse_norm_csr(matrix):
    norms = np.sqrt(matrix.multiply(matrix).sum(axis=1))
    norms = np.asarray(norms).ravel()
    norms[norms == 0] = 1.0

    return matrix.multiply(1.0 / norms[:, None]).tocsr()

In [27]:
def batch_sparse_similarity_csr(query, index):
    query = sparse_norm_csr(query)

    # index: 19 × T
    # query.T: T × C
    # result: 19 × C
    return index @ query.T

In [28]:
tfidf_sim = batch_sparse_similarity_csr(l0_tfidf, l0_tfidf_comp)

In [29]:
scores = average_similarity(ft_sim, entity_sim, tfidf_sim)

In [30]:
l0_index = index[:19]

In [31]:
def to_records(scores, index):
    records = {}
    for i, score in enumerate(scores):
        records[index[i]] = {}
        for j, comp_score in enumerate(score):
            records[index[i]][index[j]] = round(float(comp_score), 4)
    return records

In [32]:
l0_records = to_records(scores, l0_index)

In [33]:
def plot_heatmap(sim_df, annot=True):
    # Plot a heatmap of similarities
    f, ax = plt.subplots(figsize=(100, 100))
    plt.rcParams['font.family'] = 'sans-serif'
    sns.heatmap(sim_df,
                annot=annot,
                fmt='.2f',
                cmap="magma",
                cbar=False)
    xlocs, xlabels = plt.xticks()
    plt.setp(xlabels, rotation=45, ha='right', size=20)
    plt.xlabel(None)
    ylocs, ylabels = plt.yticks()
    plt.setp(ylabels, rotation=0, ha='right', size=20)
    plt.ylabel(None)
    return ax

In [34]:
def sim_table(scores, index):
    # Calculate + shape similarities for heatmap
    sim = pd.DataFrame(scores)
    sim.index = index
    sim.sort_index(inplace=True)
    sim.columns = index
    sim = sim[sorted(sim.columns)]
    return sim

In [77]:
sim = sim_table(scores, l0_index)

In [36]:
def plot_l0_heatmap(index):
    # L0 heatmap
    # Get a vector mask for L0 fields
    mask = index
    # Get keys in correct order
    l0_keys = index
    l0_sim = sim
    plot_heatmap(l0_sim, annot=False)
    # set_title(f'Level-0 Field Similarities ({lang.upper()})')
    plt.subplots_adjust(left=.22, bottom=.20)
    plt.savefig(os.path.join(FIG_DIR, f'heatmap-level-0.png'), bbox_inches='tight')
    plt.close()

In [160]:
import os
plot_l0_heatmap(l0_index)

In [37]:
FIG_DIR = "analysis/similarity"

In [38]:
l1_fasttext_comp = field_fasttext[levels == 1]
l1_tfidf_comp = field_tfidf[levels == 1]
l1_entity_comp = field_entities[levels == 1]

In [39]:
ft_sim_l1 = cosine_similarity(l1_fasttext, l1_fasttext_comp)

In [40]:
entity_sim_l1 = cosine_similarity(l1_entity, l1_entity_comp)

In [41]:
tfidf_sim_l1 = batch_sparse_similarity_csr(l1_tfidf, l1_tfidf_comp)

In [42]:
scores_l1 = average_similarity(ft_sim_l1, entity_sim_l1, tfidf_sim_l1)

In [43]:
scores_l1

array([[1.0000004 , 0.6686283 , 0.70915747, ..., 0.67417115, 0.6683631 ,
        0.6696484 ],
       [0.6686283 , 1.        , 0.68643385, ..., 0.68399316, 0.6773593 ,
        0.7075407 ],
       [0.70915747, 0.6864338 , 1.0000001 , ..., 0.67880076, 0.69547033,
        0.69796807],
       ...,
       [0.67417103, 0.68399316, 0.67880076, ..., 1.0000004 , 0.6787285 ,
        0.6778443 ],
       [0.6683631 , 0.6773593 , 0.69547033, ..., 0.6787285 , 0.9999998 ,
        0.6784273 ],
       [0.6696484 , 0.7075407 , 0.69796807, ..., 0.6778443 , 0.6784273 ,
        0.9999998 ]], dtype=float32)

In [44]:
l1_index = index[19:19+281]

In [45]:
l1_records = to_records(scores_l1, l1_index)

In [46]:
sim_l1 = sim_table(scores_l1, l1_index)

In [47]:
sim_l1

,Accounting,Acoustics,Actuarial Science,Advertising,Aeronautics,Aerospace Engineering,Aesthetics,Agricultural Economics,Agricultural Engineering,Agricultural Science,...,Traditional Medicine,Transport Engineering,Urology,Veterinary Medicine,Virology,Visual Arts,Waste Management,Water Resource Management,Welfare Economics,Zoology
Accounting,1.000000,0.668628,0.709157,0.682641,0.670057,0.666194,0.670123,0.672095,0.667661,0.668263,...,0.668710,0.667533,0.652984,0.670687,0.632931,0.676622,0.674007,0.674171,0.668363,0.669648
Acoustics,0.668628,1.000000,0.686434,0.687926,0.714672,0.694695,0.699397,0.658713,0.674711,0.668894,...,0.683057,0.675531,0.659202,0.681225,0.670990,0.711381,0.685331,0.683993,0.677359,0.707541
Actuarial Science,0.709157,0.686434,1.000000,0.696398,0.696081,0.687679,0.688951,0.700992,0.674954,0.684807,...,0.685485,0.673476,0.659207,0.680327,0.653520,0.695591,0.684503,0.678801,0.695470,0.697968
Advertising,0.682641,0.687926,0.696398,1.000000,0.695969,0.678708,0.698449,0.667996,0.667948,0.667762,...,0.685171,0.670047,0.655170,0.679375,0.659240,0.713836,0.689424,0.678268,0.686966,0.687359
Aeronautics,0.670057,0.714672,0.696081,0.695969,1.000000,0.776281,0.698113,0.661046,0.684966,0.680614,...,0.686906,0.685055,0.655049,0.684414,0.669971,0.718391,0.686619,0.683805,0.681410,0.707843
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Visual Arts,0.676622,0.711381,0.695591,0.713836,0.718391,0.685097,0.778876,0.661274,0.671543,0.672422,...,0.707234,0.667083,0.659538,0.691096,0.669691,1.000000,0.686849,0.680020,0.678169,0.710449
Waste Management,0.674007,0.685331,0.684503,0.689424,0.686619,0.674885,0.672501,0.673515,0.676567,0.663360,...,0.675918,0.678780,0.653636,0.673222,0.662822,0.686849,1.000000,0.710796,0.681220,0.682604
Water Resource Management,0.674171,0.683993,0.678801,0.678268,0.683805,0.669457,0.671136,0.680453,0.691747,0.670930,...,0.671070,0.677496,0.646431,0.666287,0.653457,0.680020,0.710796,1.000000,0.678729,0.677844
Welfare Economics,0.668363,0.677359,0.695470,0.686966,0.681410,0.662729,0.690268,0.714166,0.652599,0.659326,...,0.662481,0.657737,0.644978,0.668834,0.656059,0.678169,0.681220,0.678729,1.000000,0.678427


In [48]:
def plot_l1_heatmap(index):
    # L0 heatmap
    # Get keys in correct order
    l0_keys = index
    l0_sim = sim_l1
    plot_heatmap(sim_l1, annot=False)
    # set_title(f'Level-1 Field Similarities ({lang.upper()})')
    plt.subplots_adjust(left=.22, bottom=.20)
    plt.savefig(os.path.join(FIG_DIR, f'heatmap-level-1.png'), bbox_inches='tight')
    plt.close()

In [199]:
plot_l1_heatmap(l1_index)

In [49]:
l0_records

{'Art': {'Art': 1.0,
  'Biology': 0.7099,
  'Business': 0.7076,
  'Chemistry': 0.7172,
  'Computer Science': 0.7042,
  'Economics': 0.7264,
  'Engineering': 0.7157,
  'Environmental Science': 0.7049,
  'Geography': 0.7144,
  'Geology': 0.7034,
  'History': 0.7471,
  'Materials Science': 0.691,
  'Mathematics': 0.7229,
  'Medicine': 0.7245,
  'Philosophy': 0.7424,
  'Physics': 0.712,
  'Political Science': 0.7004,
  'Psychology': 0.7253,
  'Sociology': 0.7319},
 'Biology': {'Art': 0.7099,
  'Biology': 1.0,
  'Business': 0.7158,
  'Chemistry': 0.7887,
  'Computer Science': 0.7046,
  'Economics': 0.7357,
  'Engineering': 0.7145,
  'Environmental Science': 0.7441,
  'Geography': 0.7129,
  'Geology': 0.7482,
  'History': 0.726,
  'Materials Science': 0.7371,
  'Mathematics': 0.7201,
  'Medicine': 0.7265,
  'Philosophy': 0.7154,
  'Physics': 0.7271,
  'Political Science': 0.6798,
  'Psychology': 0.7242,
  'Sociology': 0.7155},
 'Business': {'Art': 0.7076,
  'Biology': 0.7158,
  'Business': 1

In [203]:
import json
with open('analysis/similarity/test.py', 'w') as a:
    json.dump(l1_records, a)

In [205]:
sim_l1.to_csv('analysis/similarity/l1_cosine_similarity.csv', index=False)

In [62]:
l0_disparity = {}

In [63]:
for field in l0_records:
    l0_disparity[field] = {}
    for f2 in l0_records[field]:
        l0_disparity[field][f2] = 1 - l0_records[field][f2]

In [50]:
l1_disparity = {}

In [51]:
for field in l1_records:
    l1_disparity[field] = {}
    for f2 in l1_records[field]:
        l1_disparity[field][f2] = 1 - l1_records[field][f2]

In [78]:
sim_l0_disparity = sim.applymap(lambda x: round(float( 1 - x), 5))

In [53]:
from google.cloud import bigquery

In [54]:
client = bigquery.Client()

In [55]:
query = """with get_single_cluster_for_sample as (
  select distinct
    merged_id 
  from 
    map_of_science.cluster_assignment
  where 
   cluster_id = 1
),

get_refs as (
select distinct
  merged_id,
  ref_id
from
  literature.references
inner join
  get_single_cluster_for_sample
using
  (merged_id)
),

get_fields as (
  select distinct
    top_fields.merged_id,
    top_l0
  from
    fields_of_study_v2.top_fields
  inner join
    get_refs
  on ref_id = top_fields.merged_id
)

select
  top_l0,
  count(distinct merged_id) as field_count,
  COUNT(distinct merged_id) / SUM(COUNT(distinct merged_id)) OVER () AS field_proportion
from
  get_fields
group by top_l0
"""

In [56]:
query_job = client.query(query)

In [57]:
field_proportions = {}

In [58]:
for row in query_job:
    field_proportions[row["top_l0"]] = row["field_proportion"]

In [59]:
field_proportions

{'Political Science': 0.00015026457298028314,
 'History': 0.004443538086702658,
 'Computer Science': 0.002608163659586343,
 'Philosophy': 0.00012879820541167127,
 'Sociology': 0.00018246412433320095,
 'Physics': 0.00211443720550827,
 'Chemistry': 0.1587115886185319,
 'Environmental Science': 0.0079532891841707,
 'Business': 0.00023613004325473064,
 'Biology': 0.0634438493490324,
 'Medicine': 0.0005581255567839088,
 'Engineering': 0.011763569427599308,
 'Economics': 0.0023613004325473063,
 'Geology': 0.0018139080595477037,
 'Psychology': 0.0005795919243525207,
 'Geography': 2.1466367568611877e-05,
 'Materials Science': 0.7415878672090502,
 'Mathematics': 0.0003112623297448722,
 'Art': 0.0010303856432933702}

In [60]:
from itertools import combinations

In [64]:
rao_stirling = 0
for pair in combinations(field_proportions, 2):
    rao_stirling += field_proportions[pair[0]] * field_proportions[pair[1]] * l0_disparity[pair[0]][pair[1]]

In [65]:
# multiple by 2 because we did every ij combination and skipped all the ji combinations
rao_stirling = rao_stirling * 2

In [66]:
rao_stirling

0.10225216819698463

In [87]:
sim_test = sim_l1_disparity.reset_index()

In [88]:
sim_l1_disparity = sim_test.rename(columns={'index': 'field'})

In [89]:
sim_l1_disparity.index = l1_index

In [90]:
sim_l1_disparity

,field,Accounting,Acoustics,Actuarial Science,Advertising,Aeronautics,Aerospace Engineering,Aesthetics,Agricultural Economics,Agricultural Engineering,...,Traditional Medicine,Transport Engineering,Urology,Veterinary Medicine,Virology,Visual Arts,Waste Management,Water Resource Management,Welfare Economics,Zoology
Accounting,Accounting,-0.00000,0.33137,0.29084,0.31736,0.32994,0.33381,0.32988,0.32790,0.33234,...,0.33129,0.33247,0.34702,0.32931,0.36707,0.32338,0.32599,0.32583,0.33164,0.33035
Acoustics,Acoustics,0.33137,0.00000,0.31357,0.31207,0.28533,0.30531,0.30060,0.34129,0.32529,...,0.31694,0.32447,0.34080,0.31877,0.32901,0.28862,0.31467,0.31601,0.32264,0.29246
Actuarial Science,Actuarial Science,0.29084,0.31357,-0.00000,0.30360,0.30392,0.31232,0.31105,0.29901,0.32505,...,0.31451,0.32652,0.34079,0.31967,0.34648,0.30441,0.31550,0.32120,0.30453,0.30203
Advertising,Advertising,0.31736,0.31207,0.30360,-0.00000,0.30403,0.32129,0.30155,0.33200,0.33205,...,0.31483,0.32995,0.34483,0.32063,0.34076,0.28616,0.31058,0.32173,0.31303,0.31264
Aeronautics,Aeronautics,0.32994,0.28533,0.30392,0.30403,0.00000,0.22372,0.30189,0.33895,0.31503,...,0.31309,0.31495,0.34495,0.31559,0.33003,0.28161,0.31338,0.31620,0.31859,0.29216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Visual Arts,Visual Arts,0.32338,0.28862,0.30441,0.28616,0.28161,0.31490,0.22112,0.33873,0.32846,...,0.29277,0.33292,0.34046,0.30890,0.33031,0.00000,0.31315,0.31998,0.32183,0.28955
Waste Management,Waste Management,0.32599,0.31467,0.31550,0.31058,0.31338,0.32511,0.32750,0.32649,0.32343,...,0.32408,0.32122,0.34636,0.32678,0.33718,0.31315,0.00000,0.28920,0.31878,0.31740
Water Resource Management,Water Resource Management,0.32583,0.31601,0.32120,0.32173,0.31620,0.33054,0.32886,0.31955,0.30825,...,0.32893,0.32250,0.35357,0.33371,0.34654,0.31998,0.28920,-0.00000,0.32127,0.32216
Welfare Economics,Welfare Economics,0.33164,0.32264,0.30453,0.31303,0.31859,0.33727,0.30973,0.28583,0.34740,...,0.33752,0.34226,0.35502,0.33117,0.34394,0.32183,0.31878,0.32127,0.00000,0.32157


In [285]:
sim_l0_disparity.to_csv('analysis/similarity/l0_cosine_distance.csv', index=False)

In [286]:
with open("analysis/similarity/l0-cosine-distance.jsonl", "w") as f:
    for field in l0_disparity:
        for field2 in l0_disparity[field]:
            json_elem = {"field_a": field, "field_b": field2, "cosine_distance": l0_disparity[field][field2]}
            f.write(json.dumps(json_elem) + "\n")

In [93]:
with open("analysis/similarity/l1-cosine-distance.jsonl", "w") as f:
    for field in l1_disparity:
        for field2 in l1_disparity[field]:
            json_elem = {"field_a": field, "field_b": field2, "cosine_distance": l1_disparity[field][field2]}
            f.write(json.dumps(json_elem) + "\n")